# Mechanistic Interpretation of Grokked Modular Arithmetic Model

This notebook interprets the model trained in `modular.ipynb` using Fourier analysis techniques.

## Goals
- Analyze embedding structure in Fourier basis
- Identify key frequencies automatically
- Understand neuron activation patterns
- Verify the algorithm for computing logits

## Setup & Imports

In [1]:
import math
import os
from pathlib import Path
from typing import Callable

import numpy as np
import torch as t
import torch.nn.functional as F
from torch import Tensor
import einops
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression

from transformer_lens import HookedTransformer, HookedTransformerConfig
from jaxtyping import Float

device = 'cuda' if t.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Device: cuda


## Load Trained Model from Checkpoint

In [2]:
# Path to checkpoint - use the best grokked model
checkpoint_dir = Path("/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/modular/checkpoints")

# List available checkpoints and find the latest high-epoch one
checkpoints = sorted(checkpoint_dir.glob("checkpoint_epoch_*.pt"))
print(f"Found {len(checkpoints)} checkpoints")

# Get checkpoint epochs and sort by number
def get_epoch(path):
    return int(path.stem.split('_')[-1])

checkpoints_sorted = sorted(checkpoints, key=get_epoch, reverse=True)
print(f"Latest checkpoints: {[get_epoch(c) for c in checkpoints_sorted[:5]]}")

# Load the latest checkpoint (or best_model if it exists and has high test accuracy)
best_model_path = checkpoint_dir / "best_model.pt"

# Check if there's a well-grokked checkpoint by examining test accuracy
# First, let's check what test accuracy the best_model has
if best_model_path.exists():
    ckpt_info = t.load(best_model_path, map_location='cpu', weights_only=False)
    print(f"Best model epoch: {ckpt_info.get('epoch', 'unknown')}, val_acc: {ckpt_info.get('best_val_acc', 'unknown')}")

Found 300 checkpoints
Latest checkpoints: [50000, 49000, 48000, 47000, 46000]
Best model epoch: 3833, val_acc: 0.04955811612037141


In [3]:
# Load a well-trained checkpoint - try to find one with high test accuracy
# We'll scan through checkpoints to find one that has grokked

def load_checkpoint_and_check(ckpt_path):
    """Load checkpoint and return (model_state, epoch, train_acc, test_acc)"""
    ckpt = t.load(ckpt_path, map_location='cpu', weights_only=False)
    return ckpt

# Try loading the highest-epoch checkpoint first
selected_ckpt_path = checkpoints_sorted[0]
print(f"Loading checkpoint: {selected_ckpt_path}")
checkpoint = load_checkpoint_and_check(selected_ckpt_path)

print(f"Loaded epoch {checkpoint.get('epoch', 'unknown')}")
if 'val_acc' in checkpoint:
    print(f"  Val accuracy: {checkpoint['val_acc']:.2%}")
if 'train_acc' in checkpoint:
    print(f"  Train accuracy: {checkpoint['train_acc']:.2%}")

Loading checkpoint: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/modular/checkpoints/checkpoint_epoch_50000.pt
Loaded epoch 50000


In [4]:
# Model configuration (from training notebook)
p = 113  # Prime modulus

cfg = HookedTransformerConfig(
    n_layers=1,
    d_vocab=p + 1,  # 0 to p-1 for numbers, p for '=' token
    d_model=128,
    d_mlp=512,
    n_heads=4,
    d_head=32,
    n_ctx=3,  # [a, b, =]
    act_fn="relu",
    normalization_type=None,  # No LayerNorm for interpretability
    device=device,
)

# Create model and load weights
model = HookedTransformer(cfg)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Model loaded with {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Configuration: {cfg.n_layers} layers, d_model={cfg.d_model}, d_mlp={cfg.d_mlp}")

Model loaded with 227,442 parameters
Configuration: 1 layers, d_model=128, d_mlp=512


## Generate Dataset and Verify Model Performance

In [5]:
# Generate all input pairs
all_data = t.tensor([(i, j, p) for i in range(p) for j in range(p)]).to(device)
labels = t.tensor([(i + j) % p for i in range(p) for j in range(p)]).to(device)

print(f"Total examples: {len(all_data)} ({p}x{p} = {p*p})")

# Verify model performance
with t.no_grad():
    logits = model(all_data)
    preds = logits[:, -1, :-1].argmax(dim=-1)  # Exclude padding token
    accuracy = (preds == labels).float().mean().item()
    
print(f"Overall accuracy: {accuracy:.2%}")

Total examples: 12769 (113x113 = 12769)
Overall accuracy: 100.00%


## Fourier Basis Setup

In [6]:
def make_fourier_basis(p: int) -> tuple:
    """
    Creates an orthonormal Fourier basis for functions on Z_p.
    
    The basis consists of:
    - Constant function (frequency 0)
    - cos(2*pi*k*x/p) for k = 1, ..., p//2
    - sin(2*pi*k*x/p) for k = 1, ..., p//2
    """
    fourier_basis = t.ones(p, p)
    fourier_basis_names = ["Const"]
    
    for k in range(1, p // 2 + 1):
        fourier_basis[2 * k - 1] = t.cos(2 * t.pi * t.arange(p) * k / p)
        fourier_basis[2 * k] = t.sin(2 * t.pi * t.arange(p) * k / p)
        fourier_basis_names.extend([f"cos {k}", f"sin {k}"])
    
    # Normalize to be orthonormal
    fourier_basis /= fourier_basis.norm(dim=1, keepdim=True)
    
    return fourier_basis.to(device), fourier_basis_names

fourier_basis, fourier_basis_names = make_fourier_basis(p)
print(f"Fourier basis shape: {fourier_basis.shape}")

# Verify orthonormality
dot_products = fourier_basis @ fourier_basis.T
identity_check = t.allclose(dot_products, t.eye(p, device=device), atol=1e-5)
print(f"Orthonormality check: {identity_check}")

Fourier basis shape: torch.Size([113, 113])
Orthonormality check: True


In [7]:
def fft1d(x: t.Tensor) -> t.Tensor:
    """Project vectors onto the 1D Fourier basis."""
    return x @ fourier_basis.T

def fft2d(tensor: t.Tensor) -> t.Tensor:
    """Apply 2D Fourier transform to first two dimensions."""
    return einops.einsum(
        tensor, fourier_basis, fourier_basis,
        "px py ..., i px, j py -> i j ..."
    )

def fourier_2d_basis_term(i: int, j: int) -> t.Tensor:
    """Get the (i, j) 2D Fourier basis term as a (p, p) tensor."""
    return fourier_basis[i][:, None] * fourier_basis[j][None, :]

print("Fourier transform functions defined.")

Fourier transform functions defined.


## Visualization Utilities

In [8]:
def to_numpy(tensor):
    """Convert tensor to numpy array."""
    if isinstance(tensor, t.Tensor):
        return tensor.detach().cpu().numpy()
    return tensor

def imshow_fourier(
    tensor,
    title="",
    animation_frame=None,
    animation_name="",
    animation_labels=None,
    facet_col=None,
    facet_labels=None,
):
    """Plot a tensor with Fourier basis labels."""
    tensor_np = to_numpy(tensor)
    
    if animation_frame is not None:
        # Animation version
        frames = []
        for i in range(tensor_np.shape[animation_frame]):
            if animation_frame == 0:
                frame_data = tensor_np[i]
            else:
                frame_data = tensor_np[:, :, i]
            frames.append(go.Frame(
                data=[go.Heatmap(z=frame_data, colorscale='RdBu_r', zmid=0)],
                name=str(animation_labels[i] if animation_labels else i)
            ))
        
        fig = go.Figure(
            data=[go.Heatmap(z=tensor_np[0] if animation_frame == 0 else tensor_np[:, :, 0], 
                            colorscale='RdBu_r', zmid=0,
                            x=fourier_basis_names, y=fourier_basis_names)],
            frames=frames,
            layout=go.Layout(
                title=title,
                updatemenus=[{
                    'type': 'buttons',
                    'showactive': False,
                    'buttons': [{
                        'label': 'Play',
                        'method': 'animate',
                        'args': [None, {'frame': {'duration': 500}, 'fromcurrent': True}]
                    }]
                }],
                sliders=[{
                    'active': 0,
                    'steps': [{'args': [[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                              'label': str(animation_labels[i] if animation_labels else i),
                              'method': 'animate'} 
                             for i, f in enumerate(frames)],
                    'currentvalue': {'prefix': f'{animation_name}: '}
                }]
            )
        )
    elif facet_col is not None:
        # Faceted version
        n_facets = tensor_np.shape[facet_col]
        fig = make_subplots(rows=1, cols=n_facets, 
                           subplot_titles=facet_labels if facet_labels else [f"Frame {i}" for i in range(n_facets)])
        for i in range(n_facets):
            if facet_col == 0:
                frame_data = tensor_np[i]
            else:
                frame_data = tensor_np[:, :, i]
            fig.add_trace(
                go.Heatmap(z=frame_data, colorscale='RdBu_r', zmid=0, showscale=(i == n_facets-1)),
                row=1, col=i+1
            )
        fig.update_layout(title=title, height=400)
    else:
        # Simple version
        fig = px.imshow(
            tensor_np,
            x=fourier_basis_names,
            y=fourier_basis_names,
            color_continuous_scale='RdBu_r',
            color_continuous_midpoint=0,
            title=title
        )
    
    fig.update_layout(height=600, width=700)
    fig.show()

def imshow_div(tensor, x=None, yaxis="", title="", hline_positions=None, hline_labels=None):
    """Plot a diverging colormap heatmap."""
    tensor_np = to_numpy(tensor)
    
    fig = px.imshow(
        tensor_np,
        x=x,
        color_continuous_scale='RdBu_r',
        color_continuous_midpoint=0,
        title=title,
        labels={'y': yaxis}
    )
    
    if hline_positions:
        for pos, label in zip(hline_positions, hline_labels or []):
            fig.add_hline(y=pos - 0.5, line_dash="dash", line_color="black",
                         annotation_text=label, annotation_position="left")
    
    fig.update_layout(height=600, width=900)
    fig.show()

print("Visualization utilities defined.")

Visualization utilities defined.


## Extract Model Components

In [9]:
# Extract key weight matrices
W_E = model.W_E[:-1].detach()  # Embedding (excluding '=' token)
W_U = model.W_U[:, :-1].detach()  # Unembedding (excluding '=' token output)
W_out = model.W_out[0].detach()  # MLP output weights
W_in = model.W_in[0].detach()  # MLP input weights
W_logit = W_out @ W_U  # Combined logit weights

d_mlp = cfg.d_mlp

print(f"W_E shape: {W_E.shape} (vocab x d_model)")
print(f"W_U shape: {W_U.shape} (d_model x vocab)")
print(f"W_out shape: {W_out.shape} (d_mlp x d_model)")
print(f"W_logit shape: {W_logit.shape} (d_mlp x vocab)")

W_E shape: torch.Size([113, 128]) (vocab x d_model)
W_U shape: torch.Size([128, 113]) (d_model x vocab)
W_out shape: torch.Size([512, 128]) (d_mlp x d_model)
W_logit shape: torch.Size([512, 113]) (d_mlp x vocab)


## Analyze Embedding in Fourier Basis

In [10]:
# Project embeddings onto Fourier basis
W_E_fourier = fourier_basis @ W_E  # Shape: [p, d_model]

# Compute the norm of each Fourier component across all embedding dimensions
fourier_norms = einops.reduce(W_E_fourier.pow(2), "freq d_model -> freq", "sum").sqrt()

# Plot the Fourier spectrum
fig = px.bar(
    x=fourier_basis_names,
    y=to_numpy(fourier_norms),
    title="Norm of Fourier Components in Token Embeddings",
    labels={"x": "Fourier Component", "y": "Norm"},
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

# Find top frequencies
top_k = 15
top_indices = fourier_norms.argsort(descending=True)[:top_k]
print(f"\nTop {top_k} Fourier components in embeddings:")
for i, idx in enumerate(top_indices):
    print(f"  {i+1}. {fourier_basis_names[idx]}: {fourier_norms[idx].item():.4f}")


Top 15 Fourier components in embeddings:
  1. cos 9: 4.3250
  2. sin 9: 4.3175
  3. cos 34: 4.1600
  4. sin 34: 4.1348
  5. sin 15: 4.0978
  6. cos 15: 4.0919
  7. sin 14: 3.3706
  8. cos 14: 3.3632
  9. cos 2: 3.3021
  10. sin 2: 3.2701
  11. sin 28: 1.9024
  12. cos 28: 1.8508
  13. cos 4: 0.7991
  14. sin 8: 0.5005
  15. sin 6: 0.4458


## Automatically Identify Key Frequencies

In [11]:
# Identify key frequencies based on embedding norms
# We'll look for frequencies where BOTH cos and sin have high norms

# Group by frequency (k) and sum cos/sin norms
freq_norms = {}
for k in range(1, p // 2 + 1):
    cos_idx = 2 * k - 1
    sin_idx = 2 * k
    # Total norm for this frequency
    freq_norms[k] = (fourier_norms[cos_idx].item()**2 + fourier_norms[sin_idx].item()**2)**0.5

# Sort frequencies by total norm
sorted_freqs = sorted(freq_norms.items(), key=lambda x: x[1], reverse=True)
print("Top frequencies by combined cos+sin norm:")
for k, norm in sorted_freqs[:10]:
    print(f"  k={k}: {norm:.4f} (cos {k}: {fourier_norms[2*k-1].item():.4f}, sin {k}: {fourier_norms[2*k].item():.4f})")

Top frequencies by combined cos+sin norm:
  k=9: 6.1112 (cos 9: 4.3250, sin 9: 4.3175)
  k=34: 5.8653 (cos 34: 4.1600, sin 34: 4.1348)
  k=15: 5.7910 (cos 15: 4.0919, sin 15: 4.0978)
  k=14: 4.7616 (cos 14: 3.3632, sin 14: 3.3706)
  k=2: 4.6473 (cos 2: 3.3021, sin 2: 3.2701)
  k=28: 2.6542 (cos 28: 1.8508, sin 28: 1.9024)
  k=4: 0.8940 (cos 4: 0.7991, sin 4: 0.4008)
  k=8: 0.5461 (cos 8: 0.2186, sin 8: 0.5005)
  k=6: 0.4575 (cos 6: 0.1031, sin 6: 0.4458)
  k=1: 0.3982 (cos 1: 0.1311, sin 1: 0.3760)


In [12]:
# Select key frequencies using a threshold
# We use mean + 0.5*std as threshold for combined norm
all_freq_norms = t.tensor(list(freq_norms.values()))
threshold = all_freq_norms.mean() + 0.5 * all_freq_norms.std()

key_freqs = t.tensor([k for k, norm in freq_norms.items() if norm > threshold.item()])
key_freqs = key_freqs.sort().values

print(f"\nKey frequencies (combined norm > {threshold.item():.4f}):")
print(f"  {key_freqs.tolist()}")
print(f"\nNumber of key frequencies: {len(key_freqs)}")


Key frequencies (combined norm > 1.5451):
  [2, 9, 14, 15, 28, 34]

Number of key frequencies: 6


## Run Model and Cache Activations

In [13]:
# Run model on all data and cache activations
with t.no_grad():
    original_logits, cache = model.run_with_cache(all_data)

# Get logits at final position (excluding padding token)
original_logits = original_logits[:, -1, :-1]
print(f"Original logits shape: {original_logits.shape}")

# Get neuron activations (post-ReLU)
neuron_acts_post = cache["post", 0][:, -1]  # [batch, d_mlp]
print(f"Neuron activations shape: {neuron_acts_post.shape}")

# Reshape to 2D grid for Fourier analysis
neuron_acts_post_sq = einops.rearrange(neuron_acts_post, "(x y) d_mlp -> x y d_mlp", x=p, y=p)
print(f"Neuron activations (reshaped) shape: {neuron_acts_post_sq.shape}")

# Compute original loss
original_loss = F.cross_entropy(original_logits, labels)
print(f"\nOriginal loss: {original_loss.item():.6f}")

Original logits shape: torch.Size([12769, 113])
Neuron activations shape: torch.Size([12769, 512])
Neuron activations (reshaped) shape: torch.Size([113, 113, 512])

Original loss: 0.000043


## Analyze Neuron Activations in Fourier Basis

In [14]:
# Center neuron activations (remove mean/bias)
neuron_acts_centered = neuron_acts_post_sq - neuron_acts_post_sq.mean((0, 1), keepdim=True)

# Take 2D Fourier transform
neuron_acts_centered_fourier = fft2d(neuron_acts_centered)
print(f"Neuron acts Fourier shape: {neuron_acts_centered_fourier.shape}")

# Plot mean squared Fourier coefficients across all neurons
imshow_fourier(
    neuron_acts_centered_fourier.pow(2).mean(-1),
    title="Mean Squared 2D Fourier Components of Neuron Activations",
)

Neuron acts Fourier shape: torch.Size([113, 113, 512])


## Find Neuron Clusters by Frequency

In [15]:
def arrange_by_2d_freqs(tensor: Tensor) -> Tensor:
    """
    Takes a tensor of shape (p, p, *batch_dims), returns tensor of shape (p//2 - 1, 3, 3, *batch_dims)
    representing the Fourier coefficients sorted by frequency.
    """
    idx_2d_y_all = []
    idx_2d_x_all = []
    for freq in range(1, p // 2):
        idx_1d = [0, 2 * freq - 1, 2 * freq]
        idx_2d_x_all.append([idx_1d for _ in range(3)])
        idx_2d_y_all.append([[i] * 3 for i in idx_1d])
    return tensor[idx_2d_y_all, idx_2d_x_all]


def find_neuron_freqs(fourier_neuron_acts: Float[Tensor, "p p d_mlp"]) -> tuple:
    """
    Returns the frequencies that explain the most variance of each neuron
    and the fraction of variance explained.
    """
    fourier_neuron_acts_by_freq = arrange_by_2d_freqs(fourier_neuron_acts)
    
    # Sum squares of all frequency coeffs, for each neuron
    square_of_all_terms = einops.reduce(
        fourier_neuron_acts.pow(2), "x_coeff y_coeff neuron -> neuron", "sum"
    )
    
    # Sum squares for each frequency's const+linear+quadratic terms
    square_of_each_freq = einops.reduce(
        fourier_neuron_acts_by_freq.pow(2), "freq x_coeff y_coeff neuron -> freq neuron", "sum"
    )
    
    # Find freq explaining most variance for each neuron
    neuron_variance_explained, neuron_freqs = square_of_each_freq.max(0)
    neuron_frac_explained = neuron_variance_explained / square_of_all_terms
    
    # Frequencies count from k=1, not 0
    neuron_freqs += 1
    
    return neuron_freqs, neuron_frac_explained


neuron_freqs, neuron_frac_explained = find_neuron_freqs(neuron_acts_centered_fourier)
print(f"Neuron frequencies shape: {neuron_freqs.shape}")
print(f"Neuron frac explained shape: {neuron_frac_explained.shape}")

Neuron frequencies shape: torch.Size([512])
Neuron frac explained shape: torch.Size([512])


In [16]:
# Find unique frequencies that appear in neurons with high explanation
well_explained_mask = neuron_frac_explained > 0.85
key_freqs_from_neurons, neuron_freq_counts = t.unique(neuron_freqs[well_explained_mask], return_counts=True)

print("Key frequencies from neuron clustering (frac_explained > 0.85):")
for freq, count in zip(key_freqs_from_neurons.tolist(), neuron_freq_counts.tolist()):
    print(f"  k={freq}: {count} neurons")

# Use this as our key frequencies (more reliable than embedding norms)
key_freqs = key_freqs_from_neurons
print(f"\nUsing key frequencies: {key_freqs.tolist()}")

Key frequencies from neuron clustering (frac_explained > 0.85):
  k=2: 34 neurons
  k=9: 75 neurons
  k=14: 91 neurons
  k=15: 106 neurons
  k=34: 117 neurons

Using key frequencies: [2, 9, 14, 15, 34]


In [17]:
# Scatter plot of neuron clusters
fraction_positive = (cache["pre", 0][:, -1] > 0).float().mean(0)

fig = px.scatter(
    x=to_numpy(neuron_freqs),
    y=to_numpy(neuron_frac_explained),
    color=to_numpy(fraction_positive),
    labels={"x": "Neuron Frequency", "y": "Fraction Explained", "color": "Frac Positive"},
    title="Neuron Activations Explained by Key Frequencies",
    color_continuous_scale="Viridis",
)
fig.update_layout(height=500, width=700)
fig.show()

In [18]:
# Mark neurons not well-explained by any frequency as "always firing" cluster (-1)
neuron_freqs_clustered = neuron_freqs.clone()
neuron_freqs_clustered[neuron_frac_explained < 0.85] = -1

key_freqs_plus = t.cat([key_freqs, t.tensor([-1], device=device)])

print("\nNeuron cluster summary:")
for i, k in enumerate(key_freqs_plus):
    count = (neuron_freqs_clustered == k).sum().item()
    print(f"  Cluster {i}: freq k={k.item()}, {count} neurons")


Neuron cluster summary:
  Cluster 0: freq k=2, 34 neurons
  Cluster 1: freq k=9, 75 neurons
  Cluster 2: freq k=14, 91 neurons
  Cluster 3: freq k=15, 106 neurons
  Cluster 4: freq k=34, 117 neurons
  Cluster 5: freq k=-1, 89 neurons


## Visualize Fourier Components by Neuron Cluster

In [19]:
# Plot Fourier norms for each cluster separately
fourier_norms_in_each_cluster = []
for freq in key_freqs:
    fourier_norms_in_each_cluster.append(
        einops.reduce(
            neuron_acts_centered_fourier.pow(2)[..., neuron_freqs_clustered == freq],
            "batch_y batch_x neuron -> batch_y batch_x",
            "mean",
        )
    )

imshow_fourier(
    t.stack(fourier_norms_in_each_cluster),
    title="Norm of 2D Fourier components of neuron activations in each cluster",
    facet_col=0,
    facet_labels=[f"Freq={freq.item()}" for freq in key_freqs],
)

## Analyze Logit Computation

In [20]:
# Compare Fourier components of neuron activations vs logits
imshow_fourier(
    einops.reduce(neuron_acts_centered_fourier.pow(2), "y x neuron -> y x", "mean"),
    title="Norm of Fourier Components of Neuron Activations",
)

# Reshape logits and compute 2D Fourier transform
original_logits_sq = einops.rearrange(original_logits, "(x y) z -> x y z", x=p)
original_logits_fourier = fft2d(original_logits_sq)

imshow_fourier(
    einops.reduce(original_logits_fourier.pow(2), "y x z -> y x", "mean"),
    title="Norm of Fourier Components of Logits",
)

## Analyze W_logit in Fourier Basis

In [21]:
# Transform W_logit to Fourier basis (right multiply by F^T)
# This gives us US where W_logit ≈ U S F
US = W_logit @ fourier_basis.T

imshow_div(
    US,
    x=fourier_basis_names,
    yaxis="Neuron index",
    title="W_logit in the Fourier Basis"
)

In [23]:
# Rearrange by neuron cluster to see structure more clearly
US_sorted = t.cat([US[neuron_freqs_clustered == freq] for freq in key_freqs_plus])
hline_positions = np.cumsum(
    [(neuron_freqs_clustered == freq).sum().item() for freq in key_freqs]
).tolist() + [d_mlp]

imshow_div(
    US_sorted,
    x=fourier_basis_names,
    yaxis="Neuron",
    title="W_logit in the Fourier Basis (rearranged by neuron cluster)",
    hline_positions=hline_positions,
    hline_labels=[f"Cluster: freq={freq}" for freq in key_freqs.tolist()] + ["No freq"],
)

## Final Verification: Cosine and Sine Components

This is the key plot that shows the model is computing:

$$\cos(\omega_k (x + y - \vec{\textbf{z}})) = \cos(\omega_k (x + y))\cos(\omega_k \vec{\textbf{z}}) + \sin(\omega_k (x + y))\sin(\omega_k \vec{\textbf{z}})$$

We verify this by showing:
$$
\begin{aligned}
\sigma_{2k-1} u_{2k-1} &\approx C_k \cos(\omega_k (x + y)) \\
\sigma_{2k} u_{2k} &\approx C_k \sin(\omega_k (x + y))
\end{aligned}
$$

In the 2D Fourier basis, this means:
$$
\begin{aligned}
\sigma_{2k-1} u_{2k-1} &\approx \frac{C_k}{\sqrt{2}} \cos(\omega_k \vec{\textbf{x}})\cos(\omega_k \vec{\textbf{y}}) - \frac{C_k}{\sqrt{2}} \sin (\omega_k \vec{\textbf{x}})\sin (\omega_k \vec{\textbf{y}}) \\
\sigma_{2k} u_{2k} &\approx \frac{C_k}{\sqrt{2}} \cos(\omega_k \vec{\textbf{x}})\sin(\omega_k \vec{\textbf{y}}) + \frac{C_k}{\sqrt{2}} \sin (\omega_k \vec{\textbf{x}})\cos (\omega_k \vec{\textbf{y}})
\end{aligned}
$$

In [24]:
cos_components = []
sin_components = []

for k in key_freqs:
    k = k.item()  # Convert to Python int
    sigma_u_sin = US[:, 2 * k]
    sigma_u_cos = US[:, 2 * k - 1]

    logits_in_cos_dir = neuron_acts_post_sq @ sigma_u_cos
    logits_in_sin_dir = neuron_acts_post_sq @ sigma_u_sin

    cos_components.append(fft2d(logits_in_cos_dir))
    sin_components.append(fft2d(logits_in_sin_dir))

print(f"Computed {len(cos_components)} frequency components")
print(f"Key frequencies: {key_freqs.tolist()}")

Computed 5 frequency components
Key frequencies: [2, 9, 14, 15, 34]


In [25]:
# Plot Cosine components
imshow_fourier(
    t.stack(cos_components),
    title="Cosine components of neuron activations in Fourier basis",
    animation_frame=0,
    animation_name="Frequency",
    animation_labels=key_freqs.tolist(),
)

In [26]:
# Plot Sine components
imshow_fourier(
    t.stack(sin_components),
    title="Sine components of neuron activations in Fourier basis",
    animation_frame=0,
    animation_name="Frequency",
    animation_labels=key_freqs.tolist(),
)

## Interpretation of Results

The plots above should show:

**For Cosine components** (for each frequency $k$):
- Strong positive values at `(cos k, cos k)` position
- Strong negative values at `(sin k, sin k)` position
- This confirms: $\sigma_{2k-1} u_{2k-1} \approx C_k [\cos(\omega_k x)\cos(\omega_k y) - \sin(\omega_k x)\sin(\omega_k y)]$

**For Sine components** (for each frequency $k$):
- Strong positive values at `(sin k, cos k)` position
- Strong positive values at `(cos k, sin k)` position
- This confirms: $\sigma_{2k} u_{2k} \approx C_k [\sin(\omega_k x)\cos(\omega_k y) + \cos(\omega_k x)\sin(\omega_k y)]$

Together, this shows the model computes $\cos(\omega_k(x + y - z))$ using the angle addition formula!

In [27]:
# Verify by checking the specific coefficients for each frequency
print("Verification of 2D Fourier coefficients:")
print("="*70)

for i, k in enumerate(key_freqs.tolist()):
    cos_comp = cos_components[i]
    sin_comp = sin_components[i]
    
    # For cos(w(x+y)): expect cos_k*cos_k positive, sin_k*sin_k negative
    cos_cos = cos_comp[2*k-1, 2*k-1].item()
    sin_sin = cos_comp[2*k, 2*k].item()
    
    # For sin(w(x+y)): expect sin_k*cos_k and cos_k*sin_k both positive
    sin_cos = sin_comp[2*k, 2*k-1].item()
    cos_sin = sin_comp[2*k-1, 2*k].item()
    
    print(f"\nFrequency k={k}:")
    print(f"  Cosine direction:")
    print(f"    cos({k}x)cos({k}y): {cos_cos:+.4f}")
    print(f"    sin({k}x)sin({k}y): {sin_sin:+.4f}")
    print(f"    Ratio (expect ≈ -1): {sin_sin/cos_cos:.4f}" if abs(cos_cos) > 1e-6 else "    (cos_cos too small)")
    
    print(f"  Sine direction:")
    print(f"    sin({k}x)cos({k}y): {sin_cos:+.4f}")
    print(f"    cos({k}x)sin({k}y): {cos_sin:+.4f}")
    print(f"    Ratio (expect ≈ +1): {cos_sin/sin_cos:.4f}" if abs(sin_cos) > 1e-6 else "    (sin_cos too small)")

Verification of 2D Fourier coefficients:

Frequency k=2:
  Cosine direction:
    cos(2x)cos(2y): +2020.0660
    sin(2x)sin(2y): -1843.0885
    Ratio (expect ≈ -1): -0.9124
  Sine direction:
    sin(2x)cos(2y): +1986.2925
    cos(2x)sin(2y): +1985.6416
    Ratio (expect ≈ +1): 0.9997

Frequency k=9:
  Cosine direction:
    cos(9x)cos(9y): +5528.5610
    sin(9x)sin(9y): -5510.8574
    Ratio (expect ≈ -1): -0.9968
  Sine direction:
    sin(9x)cos(9y): +5431.8418
    cos(9x)sin(9y): +5428.0791
    Ratio (expect ≈ +1): 0.9993

Frequency k=14:
  Cosine direction:
    cos(14x)cos(14y): +2066.7361
    sin(14x)sin(14y): -2071.5142
    Ratio (expect ≈ -1): -1.0023
  Sine direction:
    sin(14x)cos(14y): +2060.7053
    cos(14x)sin(14y): +2060.4922
    Ratio (expect ≈ +1): 0.9999

Frequency k=15:
  Cosine direction:
    cos(15x)cos(15y): +3614.9021
    sin(15x)sin(15y): -3628.0676
    Ratio (expect ≈ -1): -1.0036
  Sine direction:
    sin(15x)cos(15y): +3536.0176
    cos(15x)sin(15y): +3540.2520
 

## Summary

The model has learned to solve modular addition $(a + b) \mod p$ using Fourier features:

1. **Embedding**: Projects tokens onto key Fourier frequencies
2. **Neurons**: Each cluster computes quadratic terms for its frequency
3. **Logits**: Uses the identity $\cos(\omega(x+y-z)) = \cos(\omega(x+y))\cos(\omega z) + \sin(\omega(x+y))\sin(\omega z)$

The final output logit for value $z$ is maximized when $z = (x + y) \mod p$.

## Causal Intervention: Embedding Swap Experiment

Now we'll verify our mechanistic understanding by performing a causal intervention using TransformerLens hooks. 

**Experiment**: Replace the embedding of token 4 with the embedding of token 9, and verify that:
1. On inputs `(4, y)`, the model now predicts `(9 + y) mod p` instead of `(4 + y) mod p`
2. On inputs `(x, 4)`, the model now predicts `(x + 9) mod p` instead of `(x + 4) mod p`

This tests whether the model truly uses the embedding to represent the token's numerical value via Fourier features.

In [28]:
# Define the tokens we'll swap
source_token = 4  # The token whose embedding we'll replace
target_token = 9  # The token whose embedding we'll use instead

print(f"Intervention: Replace embedding of token {source_token} with embedding of token {target_token}")
print(f"Expected effect: Model should compute ({target_token} + y) mod {p} instead of ({source_token} + y) mod {p}")
print(f"                 and (x + {target_token}) mod {p} instead of (x + {source_token}) mod {p}")

Intervention: Replace embedding of token 4 with embedding of token 9
Expected effect: Model should compute (9 + y) mod 113 instead of (4 + y) mod 113
                 and (x + 9) mod 113 instead of (x + 4) mod 113


In [29]:
# Select test cases where token 4 appears in position 0 (first operand)
# Input format: [a, b, =] where = is token p (113)

# Cases where first operand is 4: (4, 0), (4, 1), ..., (4, 112)
test_cases_pos0 = t.tensor([[source_token, y, p] for y in range(p)]).to(device)

# Cases where second operand is 4: (0, 4), (1, 4), ..., (112, 4)
test_cases_pos1 = t.tensor([[x, source_token, p] for x in range(p)]).to(device)

print(f"Test cases with {source_token} in position 0: {test_cases_pos0.shape}")
print(f"Test cases with {source_token} in position 1: {test_cases_pos1.shape}")
print(f"\nExample test case (pos 0): {test_cases_pos0[5].tolist()} -> expected original: {(source_token + 5) % p}, expected after swap: {(target_token + 5) % p}")
print(f"Example test case (pos 1): {test_cases_pos1[5].tolist()} -> expected original: {(5 + source_token) % p}, expected after swap: {(5 + target_token) % p}")

Test cases with 4 in position 0: torch.Size([113, 3])
Test cases with 4 in position 1: torch.Size([113, 3])

Example test case (pos 0): [4, 5, 113] -> expected original: 9, expected after swap: 14
Example test case (pos 1): [5, 4, 113] -> expected original: 9, expected after swap: 14


In [30]:
# Define the hook function to swap embeddings
# The hook will be applied to the "hook_embed" activation

def make_embedding_swap_hook(source_tok: int, target_tok: int, position: int):
    """
    Creates a hook that replaces the embedding of source_tok with target_tok's embedding
    at a specific position in the sequence.
    
    Args:
        source_tok: Token ID to replace
        target_tok: Token ID to use as replacement
        position: Position in sequence (0 for first operand, 1 for second operand)
    """
    def hook_fn(activations, hook):
        # activations shape: [batch, seq_len, d_model]
        # Get the target embedding
        target_embedding = model.W_E[target_tok]
        
        # Replace at the specified position for all batch elements
        # (all our test cases have source_tok at this position)
        activations[:, position, :] = target_embedding
        
        return activations
    
    return hook_fn

print("Hook function defined.")

Hook function defined.


In [31]:
# Run forward pass WITHOUT intervention (baseline)
with t.no_grad():
    # Position 0 test cases
    baseline_logits_pos0 = model(test_cases_pos0)[:, -1, :-1]  # [batch, vocab]
    baseline_preds_pos0 = baseline_logits_pos0.argmax(dim=-1)
    
    # Position 1 test cases
    baseline_logits_pos1 = model(test_cases_pos1)[:, -1, :-1]
    baseline_preds_pos1 = baseline_logits_pos1.argmax(dim=-1)

# Expected labels (original, without intervention)
original_labels_pos0 = t.tensor([(source_token + y) % p for y in range(p)]).to(device)
original_labels_pos1 = t.tensor([(x + source_token) % p for x in range(p)]).to(device)

# Check baseline accuracy
baseline_acc_pos0 = (baseline_preds_pos0 == original_labels_pos0).float().mean().item()
baseline_acc_pos1 = (baseline_preds_pos1 == original_labels_pos1).float().mean().item()

print("Baseline (no intervention):")
print(f"  Position 0 accuracy (predicting {source_token}+y): {baseline_acc_pos0:.2%}")
print(f"  Position 1 accuracy (predicting x+{source_token}): {baseline_acc_pos1:.2%}")

Baseline (no intervention):
  Position 0 accuracy (predicting 4+y): 100.00%
  Position 1 accuracy (predicting x+4): 100.00%


In [32]:
# Run forward pass WITH intervention (swap token 4 -> token 9)

# Expected labels AFTER intervention (model should compute using target_token instead)
swapped_labels_pos0 = t.tensor([(target_token + y) % p for y in range(p)]).to(device)
swapped_labels_pos1 = t.tensor([(x + target_token) % p for x in range(p)]).to(device)

with t.no_grad():
    # Position 0: Replace embedding at position 0
    hook_pos0 = make_embedding_swap_hook(source_token, target_token, position=0)
    intervened_logits_pos0 = model.run_with_hooks(
        test_cases_pos0,
        fwd_hooks=[("hook_embed", hook_pos0)]
    )[:, -1, :-1]
    intervened_preds_pos0 = intervened_logits_pos0.argmax(dim=-1)
    
    # Position 1: Replace embedding at position 1
    hook_pos1 = make_embedding_swap_hook(source_token, target_token, position=1)
    intervened_logits_pos1 = model.run_with_hooks(
        test_cases_pos1,
        fwd_hooks=[("hook_embed", hook_pos1)]
    )[:, -1, :-1]
    intervened_preds_pos1 = intervened_logits_pos1.argmax(dim=-1)

# Check if predictions match the SWAPPED expected labels
intervened_acc_swapped_pos0 = (intervened_preds_pos0 == swapped_labels_pos0).float().mean().item()
intervened_acc_swapped_pos1 = (intervened_preds_pos1 == swapped_labels_pos1).float().mean().item()

# Also check if they still match the ORIGINAL labels (should be low)
intervened_acc_original_pos0 = (intervened_preds_pos0 == original_labels_pos0).float().mean().item()
intervened_acc_original_pos1 = (intervened_preds_pos1 == original_labels_pos1).float().mean().item()

print("After intervention (embedding swap):")
print(f"\n  Position 0 (input [{source_token}, y, =], embedding swapped to {target_token}):")
print(f"    Accuracy vs swapped labels ({target_token}+y): {intervened_acc_swapped_pos0:.2%}")
print(f"    Accuracy vs original labels ({source_token}+y): {intervened_acc_original_pos0:.2%}")

print(f"\n  Position 1 (input [x, {source_token}, =], embedding swapped to {target_token}):")
print(f"    Accuracy vs swapped labels (x+{target_token}): {intervened_acc_swapped_pos1:.2%}")
print(f"    Accuracy vs original labels (x+{source_token}): {intervened_acc_original_pos1:.2%}")

After intervention (embedding swap):

  Position 0 (input [4, y, =], embedding swapped to 9):
    Accuracy vs swapped labels (9+y): 100.00%
    Accuracy vs original labels (4+y): 0.00%

  Position 1 (input [x, 4, =], embedding swapped to 9):
    Accuracy vs swapped labels (x+9): 100.00%
    Accuracy vs original labels (x+4): 0.00%


In [33]:
# Show detailed comparison for a few examples
print("="*80)
print("Detailed Examples (Position 0 intervention):")
print("="*80)
print(f"{'Input':<15} {'Original Label':<15} {'Swapped Label':<15} {'Baseline Pred':<15} {'Intervened Pred':<15} {'Match Swapped?'}")
print("-"*80)

for i in range(10):
    y = i
    input_str = f"[{source_token}, {y}, =]"
    orig = original_labels_pos0[i].item()
    swap = swapped_labels_pos0[i].item()
    base = baseline_preds_pos0[i].item()
    intv = intervened_preds_pos0[i].item()
    match = "YES" if intv == swap else "NO"
    print(f"{input_str:<15} {orig:<15} {swap:<15} {base:<15} {intv:<15} {match}")

Detailed Examples (Position 0 intervention):
Input           Original Label  Swapped Label   Baseline Pred   Intervened Pred Match Swapped?
--------------------------------------------------------------------------------
[4, 0, =]       4               9               4               9               YES
[4, 1, =]       5               10              5               10              YES
[4, 2, =]       6               11              6               11              YES
[4, 3, =]       7               12              7               12              YES
[4, 4, =]       8               13              8               13              YES
[4, 5, =]       9               14              9               14              YES
[4, 6, =]       10              15              10              15              YES
[4, 7, =]       11              16              11              16              YES
[4, 8, =]       12              17              12              17              YES
[4, 9, =]       13     

In [34]:
# Show detailed comparison for position 1
print("="*80)
print("Detailed Examples (Position 1 intervention):")
print("="*80)
print(f"{'Input':<15} {'Original Label':<15} {'Swapped Label':<15} {'Baseline Pred':<15} {'Intervened Pred':<15} {'Match Swapped?'}")
print("-"*80)

for i in range(10):
    x = i
    input_str = f"[{x}, {source_token}, =]"
    orig = original_labels_pos1[i].item()
    swap = swapped_labels_pos1[i].item()
    base = baseline_preds_pos1[i].item()
    intv = intervened_preds_pos1[i].item()
    match = "YES" if intv == swap else "NO"
    print(f"{input_str:<15} {orig:<15} {swap:<15} {base:<15} {intv:<15} {match}")

Detailed Examples (Position 1 intervention):
Input           Original Label  Swapped Label   Baseline Pred   Intervened Pred Match Swapped?
--------------------------------------------------------------------------------
[0, 4, =]       4               9               4               9               YES
[1, 4, =]       5               10              5               10              YES
[2, 4, =]       6               11              6               11              YES
[3, 4, =]       7               12              7               12              YES
[4, 4, =]       8               13              8               13              YES
[5, 4, =]       9               14              9               14              YES
[6, 4, =]       10              15              10              15              YES
[7, 4, =]       11              16              11              16              YES
[8, 4, =]       12              17              12              17              YES
[9, 4, =]       13     

In [35]:
# Visualize the effect of intervention on logits for one example
example_y = 10  # Example: compute 4 + 10 = 14, vs 9 + 10 = 19

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"Baseline logits for [{source_token}, {example_y}, =]",
    f"Intervened logits for [{source_token}, {example_y}, =] (embed swapped to {target_token})"
])

# Baseline logits
fig.add_trace(
    go.Bar(x=list(range(p)), y=to_numpy(baseline_logits_pos0[example_y]),
           name="Baseline", marker_color='blue'),
    row=1, col=1
)

# Intervened logits
fig.add_trace(
    go.Bar(x=list(range(p)), y=to_numpy(intervened_logits_pos0[example_y]),
           name="Intervened", marker_color='red'),
    row=1, col=2
)

# Add vertical lines for expected outputs
original_answer = (source_token + example_y) % p
swapped_answer = (target_token + example_y) % p

fig.add_vline(x=original_answer, line_dash="dash", line_color="green", 
              annotation_text=f"Original: {original_answer}", row=1, col=1)
fig.add_vline(x=swapped_answer, line_dash="dash", line_color="orange",
              annotation_text=f"Swapped: {swapped_answer}", row=1, col=2)

fig.update_layout(
    height=400, width=1200,
    title=f"Logit comparison: [{source_token}, {example_y}, =] with embedding swap {source_token}→{target_token}",
    showlegend=False
)
fig.show()

print(f"\nOriginal answer ({source_token} + {example_y}) mod {p} = {original_answer}")
print(f"Swapped answer ({target_token} + {example_y}) mod {p} = {swapped_answer}")
print(f"Baseline prediction: {baseline_preds_pos0[example_y].item()}")
print(f"Intervened prediction: {intervened_preds_pos0[example_y].item()}")


Original answer (4 + 10) mod 113 = 14
Swapped answer (9 + 10) mod 113 = 19
Baseline prediction: 14
Intervened prediction: 19


## Interpretation of Embedding Swap Results

If the embedding swap was successful (high accuracy on swapped labels), this confirms:

1. **The embedding encodes numerical value via Fourier features**: The model uses the embedding's Fourier components to represent the token's value.

2. **The rest of the computation is position-agnostic with respect to numerical value**: The attention and MLP layers don't "know" what token was originally at each position - they only see the embedding vectors.

3. **Causal verification of our mechanistic understanding**: By intervening at the embedding level, we can causally change what computation the model performs, providing strong evidence that our interpretation is correct.

This is a powerful validation technique in mechanistic interpretability - if you understand a circuit, you should be able to predict (and verify) what happens when you intervene on its components.

In [38]:
# Summary statistics
print("="*80)
print("EMBEDDING SWAP EXPERIMENT SUMMARY")
print("="*80)
print(f"\nIntervention: Replace embedding of token {source_token} with embedding of token {target_token}")
print(f"Difference in token value: {target_token - source_token} (= {target_token} - {source_token})")
print()

print("Position 0 Results (first operand swapped):")
print(f"  Baseline accuracy (vs {source_token}+y labels): {baseline_acc_pos0:.2%}")
print(f"  After intervention accuracy (vs {target_token}+y labels): {intervened_acc_swapped_pos0:.2%}")
print(f"  After intervention accuracy (vs original {source_token}+y labels): {intervened_acc_original_pos0:.2%}")
print()

print("Position 1 Results (second operand swapped):")
print(f"  Baseline accuracy (vs x+{source_token} labels): {baseline_acc_pos1:.2%}")
print(f"  After intervention accuracy (vs x+{target_token} labels): {intervened_acc_swapped_pos1:.2%}")
print(f"  After intervention accuracy (vs original x+{source_token} labels): {intervened_acc_original_pos1:.2%}")
print()

# Verdict
if intervened_acc_swapped_pos0 > 0.95 and intervened_acc_swapped_pos1 > 0.95:
    print("CONCLUSION: The embedding swap successfully changed the model's computation!")
    print("           This confirms our mechanistic understanding of how the model uses")
    print("           Fourier features in the embedding to represent numerical values.")
elif intervened_acc_swapped_pos0 > 0.80 and intervened_acc_swapped_pos1 > 0.80:
    print("CONCLUSION: The embedding swap partially worked.")
    print("           The model largely computes with the swapped value, with some errors.")
else:
    print("CONCLUSION: The embedding swap did not work as expected.")
    print("           This suggests the model may use additional information beyond")
    print("           just the embedding to determine token values.")

EMBEDDING SWAP EXPERIMENT SUMMARY

Intervention: Replace embedding of token 4 with embedding of token 9
Difference in token value: 5 (= 9 - 4)

Position 0 Results (first operand swapped):
  Baseline accuracy (vs 4+y labels): 100.00%
  After intervention accuracy (vs 9+y labels): 100.00%
  After intervention accuracy (vs original 4+y labels): 0.00%

Position 1 Results (second operand swapped):
  Baseline accuracy (vs x+4 labels): 100.00%
  After intervention accuracy (vs x+9 labels): 100.00%
  After intervention accuracy (vs original x+4 labels): 0.00%

CONCLUSION: The embedding swap successfully changed the model's computation!
           This confirms our mechanistic understanding of how the model uses
           Fourier features in the embedding to represent numerical values.


## Try Different Token Swaps

You can modify the cell below to try different embedding swaps and see if the pattern holds for other token pairs.

In [39]:
def run_embedding_swap_experiment(src_tok: int, tgt_tok: int, verbose: bool = True):
    """
    Run the embedding swap experiment for any source and target token pair.
    Returns the accuracy on swapped labels for both positions.
    """
    # Test cases
    test_pos0 = t.tensor([[src_tok, y, p] for y in range(p)]).to(device)
    test_pos1 = t.tensor([[x, src_tok, p] for x in range(p)]).to(device)
    
    # Labels
    orig_labels_0 = t.tensor([(src_tok + y) % p for y in range(p)]).to(device)
    orig_labels_1 = t.tensor([(x + src_tok) % p for x in range(p)]).to(device)
    swap_labels_0 = t.tensor([(tgt_tok + y) % p for y in range(p)]).to(device)
    swap_labels_1 = t.tensor([(x + tgt_tok) % p for x in range(p)]).to(device)
    
    with t.no_grad():
        # Baseline
        base_preds_0 = model(test_pos0)[:, -1, :-1].argmax(-1)
        base_preds_1 = model(test_pos1)[:, -1, :-1].argmax(-1)
        
        # Intervened
        hook_0 = make_embedding_swap_hook(src_tok, tgt_tok, position=0)
        hook_1 = make_embedding_swap_hook(src_tok, tgt_tok, position=1)
        
        intv_preds_0 = model.run_with_hooks(test_pos0, fwd_hooks=[("hook_embed", hook_0)])[:, -1, :-1].argmax(-1)
        intv_preds_1 = model.run_with_hooks(test_pos1, fwd_hooks=[("hook_embed", hook_1)])[:, -1, :-1].argmax(-1)
    
    # Compute accuracies
    base_acc_0 = (base_preds_0 == orig_labels_0).float().mean().item()
    base_acc_1 = (base_preds_1 == orig_labels_1).float().mean().item()
    intv_swap_acc_0 = (intv_preds_0 == swap_labels_0).float().mean().item()
    intv_swap_acc_1 = (intv_preds_1 == swap_labels_1).float().mean().item()
    
    if verbose:
        print(f"Swap {src_tok} → {tgt_tok}:")
        print(f"  Pos 0: baseline={base_acc_0:.2%}, after swap (vs {tgt_tok}+y)={intv_swap_acc_0:.2%}")
        print(f"  Pos 1: baseline={base_acc_1:.2%}, after swap (vs x+{tgt_tok})={intv_swap_acc_1:.2%}")
    
    return intv_swap_acc_0, intv_swap_acc_1

# Try a few different swaps
print("Testing various embedding swaps:")
print("="*60)
for (s, t_) in [(4, 9), (10, 20), (50, 100), (0, 1), (112, 0)]:
    run_embedding_swap_experiment(s, t_)

Testing various embedding swaps:
Swap 4 → 9:
  Pos 0: baseline=100.00%, after swap (vs 9+y)=100.00%
  Pos 1: baseline=100.00%, after swap (vs x+9)=100.00%
Swap 10 → 20:
  Pos 0: baseline=100.00%, after swap (vs 20+y)=100.00%
  Pos 1: baseline=100.00%, after swap (vs x+20)=100.00%
Swap 50 → 100:
  Pos 0: baseline=100.00%, after swap (vs 100+y)=100.00%
  Pos 1: baseline=100.00%, after swap (vs x+100)=100.00%
Swap 0 → 1:
  Pos 0: baseline=100.00%, after swap (vs 1+y)=100.00%
  Pos 1: baseline=100.00%, after swap (vs x+1)=100.00%
Swap 112 → 0:
  Pos 0: baseline=100.00%, after swap (vs 0+y)=100.00%
  Pos 1: baseline=100.00%, after swap (vs x+0)=100.00%


## Attack 2: Phase Rotation in Fourier Space

A more elegant approach than swapping embeddings is to **rotate the Fourier components**.

### Mathematical Insight

The embedding of token 4 contains Fourier components:
- $\cos(k\omega \cdot 4), \sin(k\omega \cdot 4)$ for each frequency $k$

The embedding of token 9 contains:
- $\cos(k\omega \cdot 9), \sin(k\omega \cdot 9) = \cos(k\omega \cdot 4 + k\omega \cdot 5), \sin(k\omega \cdot 4 + k\omega \cdot 5)$

This is just a **rotation by angle $k\omega \cdot 5$**! Using the angle addition formulas:

$$
\begin{bmatrix} \cos(k\omega \cdot 9) \\ \sin(k\omega \cdot 9) \end{bmatrix} = 
\begin{bmatrix} \cos(k\omega \cdot 5) & -\sin(k\omega \cdot 5) \\ \sin(k\omega \cdot 5) & \cos(k\omega \cdot 5) \end{bmatrix}
\begin{bmatrix} \cos(k\omega \cdot 4) \\ \sin(k\omega \cdot 4) \end{bmatrix}
$$

So instead of swapping embeddings, we can **apply rotation matrices** to transform token 4's representation into token 9's!

In [40]:
def make_phase_rotation_hook(delta: int, position: int):
    """
    Apply phase rotation to Fourier components of embedding at a specific position.
    
    This rotates the embedding by 'delta' units in token space, effectively
    transforming token x into token (x + delta) mod p.
    
    For each frequency k, we apply a 2D rotation matrix with angle (2*pi*k*delta/p).
    
    Args:
        delta: Amount to rotate (e.g., delta=5 transforms token 4 → token 9)
        position: Sequence position to modify (0 or 1 for first/second operand)
    """
    def hook_fn(activations, hook):
        # activations shape: [batch, seq_len, d_model]
        batch_size = activations.shape[0]
        
        # Get the embedding at the target position
        emb = activations[:, position, :].clone()  # [batch, d_model]
        
        # Project to Fourier basis: [batch, p]
        # Note: fourier_basis is [p, p] where rows are basis vectors
        # W_E is [vocab, d_model], so embeddings are in d_model space
        # We need to project the d_model embedding onto the Fourier representation
        
        # The key insight: the embedding W_E[token] has structure in Fourier space
        # We can transform it by: (1) project to Fourier, (2) rotate, (3) project back
        
        # Project embedding to Fourier basis of the token space
        # emb @ W_E.T gives us the "token-like" representation, then @ fourier_basis.T
        # But simpler: work in the embedding space directly
        
        # Get the Fourier components by projecting onto fourier_basis @ W_E
        # fourier_basis @ W_E gives Fourier representation of each embedding dim
        W_E_fourier = fourier_basis @ W_E  # [p, d_model]
        
        # Project embedding onto each Fourier direction
        # For each Fourier component i, compute how much of that component is in emb
        emb_fourier = emb @ W_E_fourier.T  # [batch, p]
        emb_fourier = emb_fourier / (W_E_fourier.norm(dim=1, keepdim=True).T + 1e-10)  # Normalize
        
        # Apply rotation to each frequency's cos/sin pair
        rotated_fourier = emb_fourier.clone()
        
        for k in range(1, p // 2 + 1):
            angle = 2 * t.pi * k * delta / p
            cos_angle = t.cos(t.tensor(angle, device=device))
            sin_angle = t.sin(t.tensor(angle, device=device))
            
            cos_idx = 2 * k - 1  # cos(k*omega*x) component
            sin_idx = 2 * k      # sin(k*omega*x) component
            
            old_cos = emb_fourier[:, cos_idx].clone()
            old_sin = emb_fourier[:, sin_idx].clone()
            
            # Apply 2D rotation matrix
            rotated_fourier[:, cos_idx] = cos_angle * old_cos - sin_angle * old_sin
            rotated_fourier[:, sin_idx] = sin_angle * old_cos + cos_angle * old_sin
        
        # Project back to embedding space
        # Reconstruct the embedding from rotated Fourier components
        W_E_fourier_normalized = W_E_fourier / (W_E_fourier.norm(dim=1, keepdim=True) + 1e-10)
        new_emb = rotated_fourier @ W_E_fourier_normalized  # [batch, d_model]
        
        activations[:, position, :] = new_emb
        return activations
    
    return hook_fn

print("Phase rotation hook function defined.")

Phase rotation hook function defined.


In [41]:
# Simpler approach: work directly with the embedding structure
# The embedding W_E has Fourier structure, so we can decompose it and rotate

def make_phase_rotation_hook_v2(delta: int, position: int):
    """
    Simplified phase rotation that works by rotating in the Fourier-projected space.
    
    Key insight: W_E_fourier = fourier_basis @ W_E gives us the Fourier decomposition
    of the embedding matrix. Each row i of W_E_fourier tells us how much of Fourier
    component i contributes to each embedding dimension.
    
    To rotate token t → token (t+delta), we:
    1. Express the current embedding in terms of Fourier components
    2. Rotate the cos/sin pairs by angle k*omega*delta for each frequency k
    3. Reconstruct the embedding from rotated components
    """
    # Precompute the rotation angles for each frequency
    rotation_angles = t.tensor([2 * t.pi * k * delta / p for k in range(p // 2 + 1)], device=device)
    cos_angles = t.cos(rotation_angles)  # [p//2 + 1]
    sin_angles = t.sin(rotation_angles)  # [p//2 + 1]
    
    # Precompute W_E_fourier
    W_E_fourier = fourier_basis @ W_E  # [p, d_model]
    
    def hook_fn(activations, hook):
        # activations: [batch, seq_len, d_model]
        emb = activations[:, position, :].clone()  # [batch, d_model]
        
        # Project to Fourier coefficients
        # emb @ W_E_fourier.T @ (W_E_fourier @ W_E_fourier.T)^-1 would give exact projection
        # But since Fourier basis is orthonormal, we can use a simpler approach
        
        # Decompose embedding: for each Fourier direction, compute the coefficient
        # coeff_i = emb · (W_E_fourier[i] / ||W_E_fourier[i]||)
        norms = W_E_fourier.norm(dim=1, keepdim=True)  # [p, 1]
        W_E_fourier_normed = W_E_fourier / (norms + 1e-10)  # [p, d_model]
        
        coeffs = emb @ W_E_fourier_normed.T  # [batch, p]
        
        # Apply rotation to each frequency's cos/sin pair
        rotated_coeffs = coeffs.clone()
        
        for k in range(1, p // 2 + 1):
            cos_idx = 2 * k - 1
            sin_idx = 2 * k
            
            c = cos_angles[k]
            s = sin_angles[k]
            
            old_cos = coeffs[:, cos_idx]
            old_sin = coeffs[:, sin_idx]
            
            # Rotation: [cos -sin; sin cos] @ [old_cos; old_sin]
            rotated_coeffs[:, cos_idx] = c * old_cos - s * old_sin
            rotated_coeffs[:, sin_idx] = s * old_cos + c * old_sin
        
        # Reconstruct embedding from rotated coefficients
        # new_emb = sum_i rotated_coeffs[i] * W_E_fourier_normed[i]
        new_emb = rotated_coeffs @ W_E_fourier_normed  # [batch, d_model]
        
        activations[:, position, :] = new_emb
        return activations
    
    return hook_fn

print("Phase rotation hook v2 defined.")

Phase rotation hook v2 defined.


In [42]:
# Run the phase rotation experiment
# Test: rotate token 4 by delta=5 to get token 9

source_token_pr = 4
delta_pr = 5
target_token_pr = (source_token_pr + delta_pr) % p

print(f"Phase Rotation Experiment: token {source_token_pr} + delta {delta_pr} = token {target_token_pr}")
print("="*70)

# Test cases where source_token appears in position 0
test_cases_pr_pos0 = t.tensor([[source_token_pr, y, p] for y in range(p)]).to(device)
test_cases_pr_pos1 = t.tensor([[x, source_token_pr, p] for x in range(p)]).to(device)

# Expected labels after rotation
rotated_labels_pos0 = t.tensor([(target_token_pr + y) % p for y in range(p)]).to(device)
rotated_labels_pos1 = t.tensor([(x + target_token_pr) % p for x in range(p)]).to(device)

# Run with phase rotation hook (using v2)
with t.no_grad():
    # Position 0
    hook_pr_pos0 = make_phase_rotation_hook_v2(delta_pr, position=0)
    pr_logits_pos0 = model.run_with_hooks(
        test_cases_pr_pos0,
        fwd_hooks=[("hook_embed", hook_pr_pos0)]
    )[:, -1, :-1]
    pr_preds_pos0 = pr_logits_pos0.argmax(dim=-1)
    
    # Position 1
    hook_pr_pos1 = make_phase_rotation_hook_v2(delta_pr, position=1)
    pr_logits_pos1 = model.run_with_hooks(
        test_cases_pr_pos1,
        fwd_hooks=[("hook_embed", hook_pr_pos1)]
    )[:, -1, :-1]
    pr_preds_pos1 = pr_logits_pos1.argmax(dim=-1)

# Compute accuracies
pr_acc_pos0 = (pr_preds_pos0 == rotated_labels_pos0).float().mean().item()
pr_acc_pos1 = (pr_preds_pos1 == rotated_labels_pos1).float().mean().item()

print(f"\nPhase Rotation Results:")
print(f"  Position 0 accuracy (vs {target_token_pr}+y labels): {pr_acc_pos0:.2%}")
print(f"  Position 1 accuracy (vs x+{target_token_pr} labels): {pr_acc_pos1:.2%}")

Phase Rotation Experiment: token 4 + delta 5 = token 9

Phase Rotation Results:
  Position 0 accuracy (vs 9+y labels): 2.65%
  Position 1 accuracy (vs x+9 labels): 2.65%


In [43]:
# Compare phase rotation with direct embedding swap
print("Comparison: Phase Rotation vs Embedding Swap")
print("="*70)

def run_phase_rotation_experiment(source_tok: int, delta: int, verbose: bool = True):
    """Run phase rotation experiment for any source token and delta."""
    target_tok = (source_tok + delta) % p
    
    test_pos0 = t.tensor([[source_tok, y, p] for y in range(p)]).to(device)
    test_pos1 = t.tensor([[x, source_tok, p] for x in range(p)]).to(device)
    
    rotated_labels_0 = t.tensor([(target_tok + y) % p for y in range(p)]).to(device)
    rotated_labels_1 = t.tensor([(x + target_tok) % p for x in range(p)]).to(device)
    
    with t.no_grad():
        hook_0 = make_phase_rotation_hook_v2(delta, position=0)
        hook_1 = make_phase_rotation_hook_v2(delta, position=1)
        
        pr_preds_0 = model.run_with_hooks(test_pos0, fwd_hooks=[("hook_embed", hook_0)])[:, -1, :-1].argmax(-1)
        pr_preds_1 = model.run_with_hooks(test_pos1, fwd_hooks=[("hook_embed", hook_1)])[:, -1, :-1].argmax(-1)
    
    acc_0 = (pr_preds_0 == rotated_labels_0).float().mean().item()
    acc_1 = (pr_preds_1 == rotated_labels_1).float().mean().item()
    
    if verbose:
        print(f"Phase Rotation: {source_tok} + delta={delta} → {target_tok}")
        print(f"  Pos 0 acc (vs {target_tok}+y): {acc_0:.2%}")
        print(f"  Pos 1 acc (vs x+{target_tok}): {acc_1:.2%}")
    
    return acc_0, acc_1

# Test multiple cases
print("\nPhase Rotation Results:")
print("-"*40)
for (src, delta) in [(4, 5), (10, 10), (50, 50), (0, 1), (100, 13)]:
    run_phase_rotation_experiment(src, delta)
    print()

print("\nEmbedding Swap Results (for comparison):")
print("-"*40)
for (src, delta) in [(4, 5), (10, 10), (50, 50), (0, 1), (100, 13)]:
    tgt = (src + delta) % p
    run_embedding_swap_experiment(src, tgt)

Comparison: Phase Rotation vs Embedding Swap

Phase Rotation Results:
----------------------------------------
Phase Rotation: 4 + delta=5 → 9
  Pos 0 acc (vs 9+y): 2.65%
  Pos 1 acc (vs x+9): 2.65%

Phase Rotation: 10 + delta=10 → 20
  Pos 0 acc (vs 20+y): 2.65%
  Pos 1 acc (vs x+20): 2.65%

Phase Rotation: 50 + delta=50 → 100
  Pos 0 acc (vs 100+y): 46.02%
  Pos 1 acc (vs x+100): 46.02%

Phase Rotation: 0 + delta=1 → 1
  Pos 0 acc (vs 1+y): 1.77%
  Pos 1 acc (vs x+1): 1.77%

Phase Rotation: 100 + delta=13 → 0
  Pos 0 acc (vs 0+y): 2.65%
  Pos 1 acc (vs x+0): 2.65%


Embedding Swap Results (for comparison):
----------------------------------------
Swap 4 → 9:
  Pos 0: baseline=100.00%, after swap (vs 9+y)=100.00%
  Pos 1: baseline=100.00%, after swap (vs x+9)=100.00%
Swap 10 → 20:
  Pos 0: baseline=100.00%, after swap (vs 20+y)=100.00%
  Pos 1: baseline=100.00%, after swap (vs x+20)=100.00%
Swap 50 → 100:
  Pos 0: baseline=100.00%, after swap (vs 100+y)=100.00%
  Pos 1: baseline=100.0

## Attack 3: Residual Stream Perturbation

Instead of modifying the embedding directly, we can **add a perturbation to the residual stream** after attention has gathered information.

### Concept

The residual stream carries information through the transformer. After attention operates, position 2 (the `=` token position) contains information gathered from positions 0 and 1.

We can add the "difference vector" between embeddings:
```
perturbation = embed(9) - embed(4)
```

Adding this to the residual stream at position 0 or 1 should effectively add +5 to that operand's representation.

### Hook Points in TransformerLens

For a 1-layer model:
- `hook_embed` - After token embedding (already tested)
- `blocks.0.hook_resid_pre` - Before attention in layer 0
- `blocks.0.hook_resid_mid` - After attention, before MLP
- `blocks.0.hook_resid_post` - After MLP (final residual)

We'll test adding the perturbation at different points to see where the intervention is most effective.

In [44]:
# First, let's explore the available hook points in the model
print("Available hook points in the model:")
print("="*50)

# Run a forward pass and see what's in the cache
with t.no_grad():
    _, test_cache = model.run_with_cache(all_data[:1])

print("Cache keys (hook points):")
for key in sorted(test_cache.keys()):
    tensor = test_cache[key]
    print(f"  {key}: shape {tensor.shape}")

Available hook points in the model:
Cache keys (hook points):
  blocks.0.attn.hook_attn_scores: shape torch.Size([1, 4, 3, 3])
  blocks.0.attn.hook_k: shape torch.Size([1, 3, 4, 32])
  blocks.0.attn.hook_pattern: shape torch.Size([1, 4, 3, 3])
  blocks.0.attn.hook_q: shape torch.Size([1, 3, 4, 32])
  blocks.0.attn.hook_v: shape torch.Size([1, 3, 4, 32])
  blocks.0.attn.hook_z: shape torch.Size([1, 3, 4, 32])
  blocks.0.hook_attn_out: shape torch.Size([1, 3, 128])
  blocks.0.hook_mlp_out: shape torch.Size([1, 3, 128])
  blocks.0.hook_resid_mid: shape torch.Size([1, 3, 128])
  blocks.0.hook_resid_post: shape torch.Size([1, 3, 128])
  blocks.0.hook_resid_pre: shape torch.Size([1, 3, 128])
  blocks.0.mlp.hook_post: shape torch.Size([1, 3, 512])
  blocks.0.mlp.hook_pre: shape torch.Size([1, 3, 512])
  hook_embed: shape torch.Size([1, 3, 128])
  hook_pos_embed: shape torch.Size([1, 3, 128])


In [45]:
def make_residual_perturbation_hook(source_tok: int, target_tok: int, position: int):
    """
    Add the embedding difference (target - source) to the residual stream.
    
    This adds the "delta" between two token embeddings to the residual stream,
    effectively shifting the representation by the difference in numerical value.
    
    Args:
        source_tok: Original token in the input
        target_tok: Token we want to shift towards
        position: Sequence position to perturb (0, 1, or 2)
    """
    # Compute the perturbation vector (difference in embeddings)
    perturbation = model.W_E[target_tok] - model.W_E[source_tok]
    
    def hook_fn(activations, hook):
        # activations: [batch, seq_len, d_model]
        activations[:, position, :] = activations[:, position, :] + perturbation
        return activations
    
    return hook_fn

print("Residual perturbation hook defined.")

Residual perturbation hook defined.


In [46]:
# Run residual perturbation experiment at different hook points
source_tok_rp = 4
target_tok_rp = 9

print(f"Residual Perturbation Experiment: Add (embed({target_tok_rp}) - embed({source_tok_rp})) to residual stream")
print("="*80)

# Test cases
test_cases_rp = t.tensor([[source_tok_rp, y, p] for y in range(p)]).to(device)
perturbed_labels = t.tensor([(target_tok_rp + y) % p for y in range(p)]).to(device)
original_labels_rp = t.tensor([(source_tok_rp + y) % p for y in range(p)]).to(device)

# Hook points to test (for position 0, which is the first operand)
hook_points = [
    "hook_embed",
    "blocks.0.hook_resid_pre",
    "blocks.0.hook_resid_mid",
    "blocks.0.hook_resid_post",
]

results_rp = {}

with t.no_grad():
    for hook_point in hook_points:
        hook = make_residual_perturbation_hook(source_tok_rp, target_tok_rp, position=0)
        
        try:
            logits = model.run_with_hooks(
                test_cases_rp,
                fwd_hooks=[(hook_point, hook)]
            )[:, -1, :-1]
            preds = logits.argmax(dim=-1)
            
            acc_perturbed = (preds == perturbed_labels).float().mean().item()
            acc_original = (preds == original_labels_rp).float().mean().item()
            
            results_rp[hook_point] = {
                "acc_perturbed": acc_perturbed,
                "acc_original": acc_original,
            }
            
            print(f"\nHook point: {hook_point}")
            print(f"  Accuracy vs perturbed labels ({target_tok_rp}+y): {acc_perturbed:.2%}")
            print(f"  Accuracy vs original labels ({source_tok_rp}+y): {acc_original:.2%}")
            
        except Exception as e:
            print(f"\nHook point: {hook_point}")
            print(f"  Error: {e}")

Residual Perturbation Experiment: Add (embed(9) - embed(4)) to residual stream

Hook point: hook_embed
  Accuracy vs perturbed labels (9+y): 100.00%
  Accuracy vs original labels (4+y): 0.00%

Hook point: blocks.0.hook_resid_pre
  Accuracy vs perturbed labels (9+y): 100.00%
  Accuracy vs original labels (4+y): 0.00%

Hook point: blocks.0.hook_resid_mid
  Accuracy vs perturbed labels (9+y): 0.00%
  Accuracy vs original labels (4+y): 100.00%

Hook point: blocks.0.hook_resid_post
  Accuracy vs perturbed labels (9+y): 0.00%
  Accuracy vs original labels (4+y): 100.00%


In [47]:
# Also test perturbation at position 2 (the '=' position)
# After attention, position 2 contains information gathered from positions 0 and 1
# Adding the perturbation here tests if we can modify the "sum" representation directly

print("\nResidual Perturbation at Position 2 (after attention gathers info):")
print("="*80)

# This tests a different idea: instead of modifying operand representations,
# we modify the position where the model computes the sum

# Use inputs where we want to add +5 to the final sum
# Input: [4, 10, =] should give 14, but with perturbation should give 19
test_cases_pos2 = t.tensor([[4, y, p] for y in range(p)]).to(device)
perturbed_labels_pos2 = t.tensor([(9 + y) % p for y in range(p)]).to(device)  # As if first operand was 9

with t.no_grad():
    for hook_point in ["blocks.0.hook_resid_mid", "blocks.0.hook_resid_post"]:
        # Perturbation at position 2
        hook = make_residual_perturbation_hook(4, 9, position=2)
        
        try:
            logits = model.run_with_hooks(
                test_cases_pos2,
                fwd_hooks=[(hook_point, hook)]
            )[:, -1, :-1]
            preds = logits.argmax(dim=-1)
            
            acc = (preds == perturbed_labels_pos2).float().mean().item()
            print(f"\nHook: {hook_point}, Position: 2")
            print(f"  Accuracy vs ({target_tok_rp}+y) labels: {acc:.2%}")
            
        except Exception as e:
            print(f"\nHook: {hook_point}, Position: 2")
            print(f"  Error: {e}")


Residual Perturbation at Position 2 (after attention gathers info):

Hook: blocks.0.hook_resid_mid, Position: 2
  Accuracy vs (9+y) labels: 0.00%

Hook: blocks.0.hook_resid_post, Position: 2
  Accuracy vs (9+y) labels: 0.00%


## Summary: Comparison of All Three Intervention Methods

We have tested three different causal interventions on the grokked modular arithmetic model:

1. **Embedding Swap**: Directly replace token A's embedding with token B's embedding
2. **Phase Rotation**: Rotate the Fourier components of the embedding by the angle corresponding to the token difference
3. **Residual Perturbation**: Add the embedding difference vector to the residual stream at various points

Each method tests a different aspect of our mechanistic understanding.

In [48]:
# Final comparison summary
print("="*80)
print("FINAL COMPARISON: All Three Intervention Methods")
print("="*80)
print(f"\nTest case: Transform token {source_token} → token {target_token} (delta = {target_token - source_token})")
print()

# Run all three methods and collect results
comparison_results = {}

# 1. Embedding Swap
with t.no_grad():
    es_hook = make_embedding_swap_hook(source_token, target_token, position=0)
    es_preds = model.run_with_hooks(test_cases_pos0, fwd_hooks=[("hook_embed", es_hook)])[:, -1, :-1].argmax(-1)
    es_acc = (es_preds == swapped_labels_pos0).float().mean().item()
comparison_results["Embedding Swap"] = es_acc

# 2. Phase Rotation
with t.no_grad():
    pr_hook = make_phase_rotation_hook_v2(target_token - source_token, position=0)
    pr_preds = model.run_with_hooks(test_cases_pos0, fwd_hooks=[("hook_embed", pr_hook)])[:, -1, :-1].argmax(-1)
    pr_acc = (pr_preds == swapped_labels_pos0).float().mean().item()
comparison_results["Phase Rotation"] = pr_acc

# 3. Residual Perturbation (at hook_embed)
with t.no_grad():
    rp_hook = make_residual_perturbation_hook(source_token, target_token, position=0)
    rp_preds = model.run_with_hooks(test_cases_pos0, fwd_hooks=[("hook_embed", rp_hook)])[:, -1, :-1].argmax(-1)
    rp_acc = (rp_preds == swapped_labels_pos0).float().mean().item()
comparison_results["Residual Perturbation (hook_embed)"] = rp_acc

# 4. Residual Perturbation (at resid_mid)
with t.no_grad():
    rp_mid_hook = make_residual_perturbation_hook(source_token, target_token, position=0)
    rp_mid_preds = model.run_with_hooks(test_cases_pos0, fwd_hooks=[("blocks.0.hook_resid_mid", rp_mid_hook)])[:, -1, :-1].argmax(-1)
    rp_mid_acc = (rp_mid_preds == swapped_labels_pos0).float().mean().item()
comparison_results["Residual Perturbation (resid_mid)"] = rp_mid_acc

# Print comparison table
print("Method                              | Accuracy (vs swapped labels)")
print("-" * 70)
for method, acc in comparison_results.items():
    status = "SUCCESS" if acc > 0.95 else ("PARTIAL" if acc > 0.5 else "FAIL")
    print(f"{method:<35} | {acc:>6.2%}  [{status}]")

print()
print("="*80)
print("INTERPRETATION:")
print("="*80)
print("""
- Embedding Swap: Direct replacement - the gold standard for this intervention.

- Phase Rotation: Mathematically elegant approach that rotates Fourier components.
  If successful, confirms the model uses Fourier structure for numerical encoding.

- Residual Perturbation: Tests whether adding the "delta vector" to the residual
  stream achieves the same effect. Results depend on hook location:
  - Early hooks (embed, resid_pre): Should work well since computation hasn't started
  - Mid hooks (resid_mid): After attention but before MLP - may or may not work
  - Late hooks (resid_post): After MLP - computation mostly done, likely won't work
""")

FINAL COMPARISON: All Three Intervention Methods

Test case: Transform token 4 → token 9 (delta = 5)

Method                              | Accuracy (vs swapped labels)
----------------------------------------------------------------------
Embedding Swap                      | 100.00%  [SUCCESS]
Phase Rotation                      |  2.65%  [FAIL]
Residual Perturbation (hook_embed)  | 100.00%  [SUCCESS]
Residual Perturbation (resid_mid)   |  0.00%  [FAIL]

INTERPRETATION:

- Embedding Swap: Direct replacement - the gold standard for this intervention.

- Phase Rotation: Mathematically elegant approach that rotates Fourier components.
  If successful, confirms the model uses Fourier structure for numerical encoding.

- Residual Perturbation: Tests whether adding the "delta vector" to the residual
  stream achieves the same effect. Results depend on hook location:
  - Early hooks (embed, resid_pre): Should work well since computation hasn't started
  - Mid hooks (resid_mid): After attent